In [ ]:
# test installiton was sucssfull
import platform
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import shap

print(f"--- Comprehensive Thesis Environment Check ---")
print(f"Python: {platform.python_version()} ✅")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-Learn (RF & Decision Trees): {sklearn.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"CatBoost: {cb.__version__}")
print(f"SHAP (TreeExplainer): {shap.__version__}")
print(f"--------------------------------------------")
print("All model architectures verified. Ready to load LycoS datasets!")

In [ ]:
# merging all datasets CSV files into 1 file
import pandas as pd
import glob
import os

print("Starting the dataset merger... (This may take a few minutes)")

# This path targets the exact folder from your last screenshot
folder_path = r"Datasets\lycos-ids2017-master\lycos-ids2017\lycos-ids2017"
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

df_list = []
for file in all_files:
    print(f"Loading {os.path.basename(file)}...")
    # We use low_memory=False so pandas safely reads the giant numbers you saw in Excel
    df = pd.read_csv(file, low_memory=False) 
    df_list.append(df)

print("\nConcatenating all days into one master dataset...")
df_2017_master = pd.concat(df_list, ignore_index=True)

# Save the final file to your main workspace
final_path = "LycoS-IDS2017.csv"
print(f"Saving to {final_path}...")
df_2017_master.to_csv(final_path, index=False)

print(f"\n✅ DONE! Total network flows: {df_2017_master.shape[0]:,}")

In [ ]:
import pandas as pd
import numpy as np

def load_and_compress(file_path):
    print(f"Loading {file_path} in chunks to save RAM...")
    chunk_list = []
    
    # Read the file 100,000 rows at a time to prevent RAM crashes
    for chunk in pd.read_csv(file_path, chunksize=100000, low_memory=False):
        # 1. Clean Column Names (strip hidden spaces)
        chunk.columns = chunk.columns.str.strip()
        
        # 2. Standardize Labels (BENIGN = 0, Attack = 1)
        if 'Label' in chunk.columns:
            chunk['Label'] = chunk['Label'].apply(lambda x: 0 if str(x).strip().upper() == 'BENIGN' else 1)
            chunk['Label'] = chunk['Label'].astype(np.int8) # Shrinks label memory by 8x!
            
        # 3. Destroy Infinities and NaNs
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(inplace=True)
        
        # 4. Extreme Memory Compression (Convert heavy 64-bit math to 32-bit math)
        float_cols = chunk.select_dtypes(include=['float64']).columns
        chunk[float_cols] = chunk[float_cols].astype(np.float32)
        
        int_cols = chunk.select_dtypes(include=['int64', 'int']).columns
        chunk[int_cols] = chunk[int_cols].astype(np.int32)
        
        # Save the cleaned, compressed chunk
        chunk_list.append(chunk)
        
    # Stitch all the small, clean chunks back into one dataset
    final_df = pd.concat(chunk_list, ignore_index=True)
    print(f"✅ {file_path} fully loaded and cleaned! Shape: {final_df.shape[0]:,} rows.")
    return final_df

print("Initiating Memory-Safe Data Pipeline...")

# ---> THIS is where df_2017 and df_2018 are officially defined! <---
df_2017 = load_and_compress("LycoS-IDS2017.csv")
print("-" * 40)
df_2018 = load_and_compress("LycoS-Unicas-IDS2018.csv")

print("\n=== Data Successfully Loaded & Cleaned ===")

In [18]:
import numpy as np

def run_health_check(df, dataset_name):
    print(f"\n=== {dataset_name} HEALTH REPORT ===")
    
    # Check 1: Hidden Spaces in Columns
    # It looks for any column name that starts or ends with a blank space
    bad_columns = [col for col in df.columns if col.startswith(' ') or col.endswith(' ')]
    if len(bad_columns) == 0:
        print("✅ 1. Hidden Spaces: PASSED (All column names are clean)")
    else:
        print(f"❌ 1. Hidden Spaces: FAILED! Found dirty columns: {bad_columns}")

    # Check 2: Binary Translation (Labels)
    # It looks at the unique values inside the 'Label' column
    if 'Label' in df.columns:
        unique_labels = df['Label'].unique()
        # Checks if the only possible answers are 0 and 1
        if set(unique_labels).issubset({0, 1}):
            print("✅ 2. Binary Labels: PASSED (Only 0s and 1s detected)")
            # Let's see the exact split of normal vs attack traffic
            print("   Traffic Split:")
            print(df['Label'].value_counts(normalize=True) * 100)
        else:
            print(f"❌ 2. Binary Labels: FAILED! Found weird labels: {unique_labels}")
    else:
        print("❌ 2. Binary Labels: FAILED! (Label column is missing!)")

    # Check 3: Corrupted Math (NaNs and Infinities)
    # Scans the entire dataframe for nulls or infinities
    has_nans = df.isnull().values.any()
    
    # We only check numeric columns for infinities to prevent errors
    numeric_df = df.select_dtypes(include=[np.number])
    has_infs = np.isinf(numeric_df).values.any()
    
    if not has_nans and not has_infs:
        print("✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)")
    else:
        print(f"❌ 3. Corrupted Math: FAILED! (NaNs found: {has_nans} | Infinities found: {has_infs})")

# Run the scan on both datasets
print("Scanning datasets... please wait.")
run_health_check(df_2017, "2017 Master Dataset")
run_health_check(df_2018, "2018 Master Dataset")

Scanning datasets... please wait.

=== 2017 Master Dataset HEALTH REPORT ===
✅ 1. Hidden Spaces: PASSED (All column names are clean)
❌ 2. Binary Labels: FAILED! (Label column is missing!)
✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)

=== 2018 Master Dataset HEALTH REPORT ===
✅ 1. Hidden Spaces: PASSED (All column names are clean)
❌ 2. Binary Labels: FAILED! (Label column is missing!)
✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)


In [ ]:
#recreating now cleand datasets 
print("Saving the perfectly clean, 1/0 datasets to your hard drive...")

# Save 2017 Clean Data
df_2017.to_csv("LycoS-IDS2017_CLEANED.csv", index=False)
print("✅ 2017 Clean Data Saved!")

# Save 2018 Clean Data (This one might take a minute since it's 13.6M rows)
df_2018.to_csv("LycoS-Unicas-IDS2018_CLEANED.csv", index=False)
print("✅ 2018 Clean Data Saved!")

print("\n🎉 SUCCESS! From now on, you never have to run the cleaning script again!")

Saving the perfectly clean, 1/0 datasets to your hard drive...
✅ 2017 Clean Data Saved!
✅ 2018 Clean Data Saved!

🎉 SUCCESS! From now on, you never have to run the cleaning script again!


In [22]:
import pandas as pd
import numpy as np

def verify_clean_data(df, dataset_name):
    print(f"\n=== {dataset_name} FINAL HEALTH CHECK ===")
    
    # 1. Verify Hidden Spaces
    bad_cols = [col for col in df.columns if col.startswith(' ') or col.endswith(' ')]
    if len(bad_cols) == 0:
        print("✅ 1. Hidden Spaces: PASSED (All column names are perfectly stripped)")
    else:
        print(f"❌ 1. Hidden Spaces: FAILED! Found dirty columns: {bad_cols}")
        
    # 2. Verify Binary Translation
    if 'label' in df.columns:
        unique_vals = df['label'].unique()
        # Checks if the only possible answers left are 0 and 1
        if set(unique_vals).issubset({0, 1}):
            print("✅ 2. Binary Translation: PASSED (Only 0s and 1s detected)")
        else:
            print(f"❌ 2. Binary Translation: FAILED! Found rogue labels: {unique_vals}")
    else:
        print("❌ 2. Binary Translation: FAILED! ('label' column is missing entirely)")
        
    # 3. Verify Corrupted Math (NaNs and Infinities)
    has_nans = df.isnull().values.any()
    numeric_df = df.select_dtypes(include=[np.number])
    has_infs = np.isinf(numeric_df).values.any()
    
    if not has_nans and not has_infs:
        print("✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)")
    else:
        print(f"❌ 3. Corrupted Math: FAILED! (NaNs found: {has_nans} | Infinities found: {has_infs})")


print("Loading your permanently saved CLEAN datasets from the hard drive...")
# ---> USING THE NEW CLEANED FILE NAMES <---
df_2017_clean = pd.read_csv("LycoS-IDS2017_CLEANED.csv")
df_2018_clean = pd.read_csv("LycoS-Unicas-IDS2018_CLEANED.csv")

print("\nInitiating Final Verification Scan on all 15.5 Million rows...")
verify_clean_data(df_2017_clean, "2017 Cleaned Dataset")
verify_clean_data(df_2018_clean, "2018 Cleaned Dataset")

Loading your permanently saved CLEAN datasets from the hard drive...

Initiating Final Verification Scan on all 15.5 Million rows...

=== 2017 Cleaned Dataset FINAL HEALTH CHECK ===
✅ 1. Hidden Spaces: PASSED (All column names are perfectly stripped)
✅ 2. Binary Translation: PASSED (Only 0s and 1s detected)
✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)

=== 2018 Cleaned Dataset FINAL HEALTH CHECK ===
✅ 1. Hidden Spaces: PASSED (All column names are perfectly stripped)
✅ 2. Binary Translation: PASSED (Only 0s and 1s detected)
✅ 3. Corrupted Math: PASSED (Zero NaNs and Zero Infinities)
